# Feature Analysis
Phân tích 4 feature ngôn ngữ để phân biệt Human vs AI text:
- **Burstiness**  độ bùng nổ từ vựng
- **Entropy**  độ hỗn loạn phân phối từ
- **TTR**  Type-Token Ratio, độ đa dạng từ vựng
- **Avg Sentence Length**  độ dài câu trung bình

In [ ]:
!pip install matplotlib seaborn scipy -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import entropy as scipy_entropy
from collections import Counter
import re

# Load data
df = pd.read_csv("processed_data/test.csv", encoding="utf-8-sig")
df["label_str"] = df["label"].map({0: "Human", 1: "AI"})
print(f"Test set: {len(df):,} records")
print(df.groupby(["language", "label_str"]).size().to_string())

In [ ]:
#  Tính 4 features 

def tokenize(text):
    return re.findall(r'\b\w+\b', text.lower())

def get_sentences(text):
    return [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]

def calc_burstiness(text):
    """
    Burstiness = (std - mean) / (std + mean) của tần suất từ.
    AI text thường đều đặn hơn  burstiness thấp hơn human.
    """
    words = tokenize(text)
    if len(words) < 10:
        return np.nan
    freq = list(Counter(words).values())
    mu, sigma = np.mean(freq), np.std(freq)
    if mu + sigma == 0:
        return 0
    return (sigma - mu) / (sigma + mu)

def calc_entropy(text):
    """
    Shannon entropy của phân phối từ.
    AI text thường có entropy thấp hơn (phân phối đều hơn, ít bất ngờ hơn).
    """
    words = tokenize(text)
    if len(words) < 10:
        return np.nan
    freq = list(Counter(words).values())
    prob = np.array(freq) / sum(freq)
    return scipy_entropy(prob)

def calc_ttr(text):
    """
    Type-Token Ratio = unique words / total words.
    AI text thường lặp từ nhiều hơn  TTR thấp hơn.
    """
    words = tokenize(text)
    if len(words) == 0:
        return np.nan
    return len(set(words)) / len(words)

def calc_avg_sent_len(text):
    """
    Độ dài câu trung bình (số từ/câu).
    AI text thường có câu dài và đều hơn.
    """
    sentences = get_sentences(text)
    if len(sentences) == 0:
        return np.nan
    lengths = [len(tokenize(s)) for s in sentences]
    return np.mean(lengths)

print(" Đang tính features...")
df["burstiness"]    = df["text"].apply(calc_burstiness)
df["entropy"]       = df["text"].apply(calc_entropy)
df["ttr"]           = df["text"].apply(calc_ttr)
df["avg_sent_len"]  = df["text"].apply(calc_avg_sent_len)
print(" Xong!")
print(df[["burstiness", "entropy", "ttr", "avg_sent_len"]].describe().round(3).to_string())

In [ ]:
#  Visualize: Boxplot 4 features × 2 ngôn ngữ 
FEATURES = {
    "burstiness":   "Burstiness",
    "entropy":      "Entropy",
    "ttr":          "TTR (Type-Token Ratio)",
    "avg_sent_len": "Avg Sentence Length",
}
LANGS    = {"en": "English", "vi": "Vietnamese"}
PALETTE  = {"Human": "#4C9BE8", "AI": "#E8734C"}

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle("Feature Distribution: Human vs AI Text", fontsize=15, fontweight="bold")

for row, (lang, lang_name) in enumerate(LANGS.items()):
    sub = df[df["language"] == lang].dropna()
    for col, (feat, feat_name) in enumerate(FEATURES.items()):
        ax = axes[row][col]
        sns.boxplot(data=sub, x="label_str", y=feat,
                    palette=PALETTE, ax=ax, width=0.5,
                    order=["Human", "AI"])
        ax.set_title(f"{feat_name}\n({lang_name})", fontsize=10)
        ax.set_xlabel("")
        ax.set_ylabel(feat_name if col == 0 else "")

plt.tight_layout()
plt.savefig("feature_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print(" Đã lưu: feature_analysis.png")

In [ ]:
#  Thống kê trung bình theo nhóm 
print(" Trung bình các feature theo ngôn ngữ và nhãn:")
summary = df.groupby(["language", "label_str"])[["burstiness", "entropy", "ttr", "avg_sent_len"]].mean().round(4)
print(summary.to_string())

# Nhận xét tự động
print("\n Nhận xét:")
for lang in ["en", "vi"]:
    sub = df[df["language"] == lang]
    for feat in ["burstiness", "entropy", "ttr", "avg_sent_len"]:
        human_mean = sub[sub["label"] == 0][feat].mean()
        ai_mean    = sub[sub["label"] == 1][feat].mean()
        direction  = "cao hơn" if human_mean > ai_mean else "thấp hơn"
        diff_pct   = abs(human_mean - ai_mean) / (abs(ai_mean) + 1e-9) * 100
        print(f"  [{lang.upper()}] Human {feat} {direction} AI ({diff_pct:.1f}%)")